# WEEK 2 — FEATURES / PREPROCESSING

## 0. Mục tiêu Week 2

Notebook này tiếp nối trực tiếp kết luận Week 1 và chỉ kiểm chứng giai đoạn Features/Preprocessing. Mục tiêu gồm:

- kế thừa cleaning của Lợi và encoding/scaling/split của Trí;
- tạo đúng 24 đặc trưng có ý nghĩa do Danh phụ trách;
- dùng đúng **một** chiến lược chọn đặc trưng;
- tích hợp pipeline do Nhân phụ trách theo thứ tự split trước, fit chỉ trên train;
- kiểm tra leakage, schema, NaN/inf và lưu artifact/dữ liệu đã xử lý.

Không huấn luyện hoặc đánh giá mô hình Week 3 trong notebook này.

## 1. Imports và đường dẫn

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / 'data' / 'raw' / 'bank.csv').is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy data/raw/bank.csv trong project.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import MissingHandler, OutlierClipper, StringCleaner
from src.encoding_scaling import (
    RANDOM_STATE,
    TARGET_COLUMN,
    get_feature_configuration_summary,
    split_data,
)
from src.feature_engineering import (
    ENGINEERED_FEATURES,
    get_feature_documentation,
)
from src.pipeline import (
    build_preprocessing_pipeline,
    get_fitted_pipeline_summary,
    load_correct_dataset,
    load_processed_splits,
    save_preprocessing_artifact,
    save_processed_splits,
    validate_transformed_splits,
)

DATA_PATH = ROOT / 'data' / 'raw' / 'bank.csv'
MODEL_PATH = ROOT / 'models' / 'preprocessing_pipeline.pkl'
PROCESSED_DIR = ROOT / 'data' / 'processed'

print(f'Project root: {ROOT}')
print(f'Dataset: {DATA_PATH}')

Project root: E:\Downloads\BTL_ML_git
Dataset: E:\Downloads\BTL_ML_git\data\raw\bank.csv


## 2. Đọc đúng dữ liệu thô

Tệp `bank.csv` là nội dung đã được xác minh từ `archive (2).zip` trong Week 1. Loader kiểm tra cả thứ tự 17 cột và hai nhãn `deposit` trước khi trả dữ liệu.

In [2]:
data = load_correct_dataset(DATA_PATH)
assert data.shape == (11162, 17)
assert data.columns[-1] == TARGET_COLUMN
assert set(data[TARGET_COLUMN].unique()) == {'no', 'yes'}

print(f'Kích thước dữ liệu: {data.shape[0]:,} dòng × {data.shape[1]} cột')
display(data.head())

Kích thước dữ liệu: 11,162 dòng × 17 cột


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


## 3. Xác định X và y

In [3]:
X = data.drop(columns=TARGET_COLUMN)
y = data[TARGET_COLUMN]

assert TARGET_COLUMN not in X.columns
assert X.shape == (11162, 16)
print('X:', X.shape)
print('y:', y.shape)
display(y.value_counts().rename('count').to_frame())

X: (11162, 16)
y: (11162,)


,count
deposit,
no,5873
yes,5289


## 4. Chia train/test trước mọi thao tác `.fit()`

Sử dụng nguyên hàm `split_data` của Trí với `test_size=0.2`, `random_state=42` và `stratify=y`. Không tạo một cách chia cạnh tranh khác.

In [4]:
X_train, X_test, y_train, y_test = split_data(
    data,
    target_column=TARGET_COLUMN,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=True,
)

assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(data)
assert TARGET_COLUMN not in X_train.columns
assert TARGET_COLUMN not in X_test.columns

class_proportions = pd.DataFrame({
    'full': y.value_counts(normalize=True),
    'train': y_train.value_counts(normalize=True),
    'test': y_test.value_counts(normalize=True),
})
print(f'X_train: {X_train.shape}; X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}; y_test: {y_test.shape}')
display(class_proportions.round(4))

X_train: (8929, 16); X_test: (2233, 16)
y_train: (8929,); y_test: (2233,)


,full,train,test
deposit,,,
no,0.5262,0.5262,0.5262
yes,0.4738,0.4738,0.4738


## 5. Chiến lược cleaning kế thừa từ Lợi

Giữ nguyên ba transformer của Lợi: chuẩn hóa chuỗi, xử lý missing ngầm định và clipping IQR. Thay đổi tương thích duy nhất là các transformer số bỏ qua cột không còn trong đầu vào, vì integration loại `duration` trước cleaning để phục vụ dự đoán **trước cuộc gọi**. Mọi median/mean/IQR vẫn được học trong `fit(X_train)`.

In [5]:
cleaning_strategy = pd.DataFrame([
    {'Bước': 'StringCleaner', 'Vai trò': "strip/lower category; admin. → admin"},
    {'Bước': 'MissingHandler', 'Vai trò': "unknown/NaN → missing; statistic số học từ train"},
    {'Bước': 'OutlierClipper', 'Vai trò': "clip theo 1,5×IQR học từ train; giữ pdays=-1"},
])
display(cleaning_strategy)

,Bước,Vai trò
0,StringCleaner,strip/lower category; admin. → admin
1,MissingHandler,unknown/NaN → missing; statistic số học từ train
2,OutlierClipper,"clip theo 1,5×IQR học từ train; giữ pdays=-1"


## 6. Chiến lược encoding/scaling kế thừa từ Trí

Giữ nguyên cấu hình của Trí: OrdinalEncoder cho `education`, OneHotEncoder cho category danh nghĩa, RobustScaler cho nhóm lệch/ngoại lai và StandardScaler cho `age`, `day`. Integration chỉ lọc `duration` khỏi nhóm RobustScaler rồi nối thêm cấu hình cho 24 feature mới.

In [6]:
display(get_feature_configuration_summary())

,Nhóm cột,Cột,Bộ chuyển đổi,Lý do
0,Biến phân loại có thứ bậc,education,OrdinalEncoder,education có thứ tự tự nhiên primary < seconda...
1,Biến phân loại danh nghĩa,"job, marital, default, housing, loan, contact,...",OneHotEncoder(handle_unknown='ignore'),Không áp đặt quan hệ thứ tự giả; bỏ qua nhãn l...
2,Biến số lệch/dao động lớn,"balance, duration, campaign, pdays, previous",RobustScaler,"Các cột như balance, duration, campaign, pdays..."
3,Biến số ổn định hơn,"age, day",StandardScaler,"age và day có miền giá trị rõ hơn, phù hợp chu..."


## 7. Danh — Feature engineering

Week 1 cho thấy `balance`, `campaign`, `previous` lệch phải; `pdays=-1` là sentinel; `month`, `poutcome`, `contact` có association đáng chú ý; dữ liệu không có ID rõ ràng. Dựa trên đó, transformer tạo đúng 24 feature dưới đây. Không feature nào dùng target hoặc `duration`.

In [7]:
feature_documentation = get_feature_documentation()
assert len(ENGINEERED_FEATURES) == 24
assert feature_documentation['feature'].is_unique
assert not feature_documentation['sources'].str.contains('duration|deposit', case=False).any()

print(f'Số feature được tạo: {len(ENGINEERED_FEATURES)}')
display(feature_documentation)

Số feature được tạo: 24


,feature,formula,sources,interpretation,rationale
0,age_group,"cut(age, [-inf, 30, 40, 60, inf))",age,"Nhóm giai đoạn tuổi: dưới 30, 30–39, 40–59, từ...",Cho phép quan hệ theo giai đoạn tuổi thay vì c...
1,age_squared,(age / 10)^2,age,Thành phần bậc hai của tuổi trên thang thập niên.,Bổ sung độ cong mà một hệ số age tuyến tính kh...
2,balance_positive_log,"log1p(max(balance, 0))",balance,Quy mô số dư dương trên thang log.,Week 1 ghi nhận balance lệch phải mạnh; log gi...
3,balance_debt_log,"log1p(max(-balance, 0))",balance,Quy mô thấu chi/số dư âm trên thang log.,Giữ riêng ý nghĩa nghiệp vụ của 688 số dư âm t...
4,has_negative_balance,1(balance < 0),balance,Cờ khách hàng có số dư âm.,Phân biệt trạng thái thấu chi với số dư không âm.
5,balance_per_age,"balance / max(age, 1)","balance, age",Số dư chuẩn hóa thô theo tuổi.,Biểu diễn tương tác giữa quy mô tài chính và g...
6,any_loan,1(housing == 'yes' or loan == 'yes'),"housing, loan",Có ít nhất một trong hai loại khoản vay.,Tóm tắt trạng thái đang có nghĩa vụ vay.
7,both_loans,1(housing == 'yes' and loan == 'yes'),"housing, loan",Đồng thời có vay mua nhà và vay cá nhân.,Nắm bắt tương tác gánh nặng từ hai khoản vay.
8,credit_obligation_count,1(default=yes) + 1(housing=yes) + 1(loan=yes),"default, housing, loan",Số trạng thái nghĩa vụ/rủi ro tín dụng đang đư...,Tổng hợp ba cờ tài chính liên quan thay vì xét...
9,financial_stress_flag,1(default == 'yes' or balance < 0),"default, balance",Có dấu hiệu default hoặc thấu chi.,Kết hợp hai tín hiệu khó khăn tài chính có ý n...


## 8. Danh — Một chiến lược feature selection

Chỉ sử dụng **Mutual Information top-K** sau encoding. Lý do: target là nhị phân, ma trận sau encoding hoàn toàn số và MI có thể nhận biết quan hệ phi tuyến/không đơn điệu mà Pearson không phản ánh. `k=32` là ngân sách kỹ thuật cố định bằng hai lần số predictor thô, không phải ngưỡng khoa học và không được tinh chỉnh trên test. Selector chỉ nhận `y_train`, dùng `random_state=42` và không bao giờ đọc `y_test`.

In [8]:
SELECTED_FEATURE_COUNT = 32
print('Chiến lược duy nhất: Mutual Information top-K')
print(f'K cố định: {SELECTED_FEATURE_COUNT}')

Chiến lược duy nhất: Mutual Information top-K
K cố định: 32


## 9. Xây dựng pipeline tích hợp

In [9]:
preprocessing_pipeline = build_preprocessing_pipeline(
    selected_feature_count=SELECTED_FEATURE_COUNT,
    random_state=RANDOM_STATE,
)
display(pd.DataFrame({
    'Thứ tự': range(1, len(preprocessing_pipeline.steps) + 1),
    'Tên bước': [name for name, _ in preprocessing_pipeline.steps],
    'Transformer': [step.__class__.__name__ for _, step in preprocessing_pipeline.steps],
}))

,Thứ tự,Tên bước,Transformer
0,1,schema_guard,PreCallSchemaGuard
1,2,string_cleaning,StringCleaner
2,3,missing_values,MissingHandler
3,4,outlier_clipping,OutlierClipper
4,5,feature_engineering,BankMarketingFeatureEngineer
5,6,encoding_scaling,ColumnTransformer
6,7,feature_selection,MutualInformationSelector


## 10. Fit CHỈ trên X_train, y_train

Dòng `.fit()` duy nhất dưới đây nhận train. X_test và y_test chưa được truyền vào bất kỳ transformer nào.

In [10]:
preprocessing_pipeline.fit(X_train, y_train)

guard = preprocessing_pipeline.named_steps['schema_guard']
missing_handler = preprocessing_pipeline.named_steps['missing_values']
selector = preprocessing_pipeline.named_steps['feature_selection']

np.testing.assert_array_equal(guard.fit_indices_, X_train.index)
assert set(guard.fit_indices_).isdisjoint(X_test.index)
assert guard.fit_row_count_ == len(X_train)
assert selector.fit_row_count_ == len(X_train)
assert selector.target_class_counts_ == y_train.value_counts().sort_index().to_dict()
assert missing_handler.fill_values_['day'] == X_train['day'].mean()
assert missing_handler.fill_values_['pdays'] == X_train.loc[X_train['pdays'] >= 0, 'pdays'].median()

print('Pipeline fit hoàn tất chỉ với train.')

Pipeline fit hoàn tất chỉ với train.


## 11. Transform train và test

In [11]:
X_train_processed = preprocessing_pipeline.transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print('Train transformed:', X_train_processed.shape)
print('Test transformed:', X_test_processed.shape)
display(X_train_processed.head())

Train transformed: (8929, 32)
Test transformed: (2233, 32)


,onehot_nominal__housing_no,onehot_nominal__housing_yes,onehot_nominal__contact_cellular,onehot_nominal__contact_missing,onehot_nominal__month_mar,onehot_nominal__month_may,onehot_nominal__month_oct,onehot_nominal__month_sep,onehot_nominal__poutcome_missing,onehot_nominal__poutcome_success,...,robust_engineered__previous_contacts_log,robust_engineered__previous_contact_intensity,robust_engineered__campaign_vs_previous_ratio,robust_engineered__successful_prior_contact_recency,robust_engineered__contact_month_sin,robust_engineered__contact_month_cos,engineered_binary__any_loan,engineered_binary__credit_obligation_count,engineered_binary__prior_contacted,engineered_binary__contact_known
8644,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.000000,0.000000,2.000000,0.000000,-0.366025,-5.773503e-01,1,1,0,1
2632,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,1.807355,10.315789,-0.714286,0.010526,-1.000000,-7.691851e-16,0,0,1,1
3056,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.000000,4.260870,-0.500000,0.010870,-0.366025,1.732051e+00,1,1,1,1
1080,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.000000,0.000000,2.000000,0.000000,-1.000000,1.154701e+00,1,1,0,1
4006,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.000000,0.000000,1.000000,0.000000,-1.000000,-7.691851e-16,0,0,0,1


## 12. Kiểm tra leakage

Các kiểm tra dưới đây xác nhận split xảy ra trước fit; fit row index không giao test; selector chỉ thấy nhãn train; target/ID lạ bị chặn; `duration` không thể ảnh hưởng output; và X_test chỉ đi qua `.transform()`.

In [12]:
assert TARGET_COLUMN not in X_train.columns
assert TARGET_COLUMN not in X_train_processed.columns
assert TARGET_COLUMN not in X_test_processed.columns

identifier_like = [
    col for col in X.columns
    if col.lower() in {'id', 'customer_id', 'user_id'}
    or col.lower().endswith('_id')
]
assert identifier_like == []

assert 'duration' not in guard.transform(X_train).columns
assert not any('duration' in str(col).lower() for col in X_train_processed.columns)

# Kết quả không đổi nếu duration bị bỏ hoặc bị thay bằng giá trị cực đoan.
sample = X_test.iloc[:50].copy()
sample_without_duration = sample.drop(columns='duration')
sample_changed_duration = sample.copy()
sample_changed_duration['duration'] = sample_changed_duration['duration'] + 1_000_000
expected_sample = preprocessing_pipeline.transform(sample)
pd.testing.assert_frame_equal(
    expected_sample, preprocessing_pipeline.transform(sample_without_duration)
)
pd.testing.assert_frame_equal(
    expected_sample, preprocessing_pipeline.transform(sample_changed_duration)
)

leakage_checks = pd.DataFrame([
    {'Kiểm tra': 'Split trước fit', 'Kết quả': 'PASS'},
    {'Kiểm tra': 'Fit indices chỉ thuộc train', 'Kết quả': 'PASS'},
    {'Kiểm tra': 'Selector chỉ nhận y_train', 'Kết quả': 'PASS'},
    {'Kiểm tra': 'Target vắng khỏi X', 'Kết quả': 'PASS'},
    {'Kiểm tra': 'Không có ID-like column', 'Kết quả': 'PASS'},
    {'Kiểm tra': 'duration không ảnh hưởng output', 'Kết quả': 'PASS'},
])
display(leakage_checks)

,Kiểm tra,Kết quả
0,Split trước fit,PASS
1,Fit indices chỉ thuộc train,PASS
2,Selector chỉ nhận y_train,PASS
3,Target vắng khỏi X,PASS
4,Không có ID-like column,PASS
5,duration không ảnh hưởng output,PASS


## 13. Schema, NaN, infinity và dimensionality

In [13]:
validation = validate_transformed_splits(
    X_train_processed,
    X_test_processed,
    expected_train_rows=len(X_train),
    expected_test_rows=len(X_test),
)
pd.testing.assert_frame_equal(
    X_test_processed,
    preprocessing_pipeline.transform(X_test),
)

feature_summary = get_fitted_pipeline_summary(preprocessing_pipeline)
dimension_summary = pd.DataFrame([
    {'Mốc': 'Predictor thô', 'Số feature': feature_summary['raw_feature_count']},
    {'Mốc': 'Sau loại duration leakage', 'Số feature': feature_summary['leakage_safe_original_count']},
    {'Mốc': 'Sau feature engineering', 'Số feature': feature_summary['after_feature_engineering_count']},
    {'Mốc': 'Sau encoding/scaling', 'Số feature': feature_summary['encoded_candidate_count']},
    {'Mốc': 'Sau MI selection', 'Số feature': feature_summary['after_feature_selection_count']},
])
display(dimension_summary)
display(pd.DataFrame([validation]))

,Mốc,Số feature
0,Predictor thô,16
1,Sau loại duration leakage,15
2,Sau feature engineering,39
3,Sau encoding/scaling,74
4,Sau MI selection,32


,train_shape,test_shape,same_schema,train_nan_count,test_nan_count,train_inf_count,test_inf_count
0,"(8929, 32)","(2233, 32)",True,0,0,0,0


## 14. Lưu preprocessing artifact

In [14]:
saved_model_path = save_preprocessing_artifact(preprocessing_pipeline, MODEL_PATH)
reloaded_pipeline = joblib.load(saved_model_path)

pd.testing.assert_frame_equal(
    preprocessing_pipeline.transform(X_test.iloc[:100]),
    reloaded_pipeline.transform(X_test.iloc[:100]),
)
print(f'Đã lưu và reload thành công: {saved_model_path}')
print(f'Kích thước artifact: {saved_model_path.stat().st_size:,} bytes')

Đã lưu và reload thành công: E:\Downloads\BTL_ML_git\models\preprocessing_pipeline.pkl
Kích thước artifact: 92,353 bytes


## 15. Lưu processed train/test

In [15]:
processed_paths = save_processed_splits(
    X_train_processed,
    X_test_processed,
    y_train,
    y_test,
    PROCESSED_DIR,
)
reloaded_splits = load_processed_splits(PROCESSED_DIR)

assert reloaded_splits['X_train'].shape == X_train_processed.shape
assert reloaded_splits['X_test'].shape == X_test_processed.shape
assert reloaded_splits['y_train'].shape == (len(y_train), 1)
assert reloaded_splits['y_test'].shape == (len(y_test), 1)
assert reloaded_splits['X_train'].columns.tolist() == X_train_processed.columns.tolist()
assert reloaded_splits['X_test'].columns.tolist() == X_test_processed.columns.tolist()
assert reloaded_splits['y_train'][TARGET_COLUMN].tolist() == y_train.reset_index(drop=True).tolist()
assert reloaded_splits['y_test'][TARGET_COLUMN].tolist() == y_test.reset_index(drop=True).tolist()

artifact_table = pd.DataFrame([
    {'Artifact': name, 'Path': str(path.relative_to(ROOT)), 'Bytes': path.stat().st_size}
    for name, path in processed_paths.items()
])
display(artifact_table)

,Artifact,Path,Bytes
0,X_train,data\processed\X_train.parquet,252747
1,X_test,data\processed\X_test.parquet,94894
2,y_train,data\processed\y_train.parquet,2321
3,y_test,data\processed\y_test.parquet,1455


## 16. Tóm tắt kỹ thuật Week 2

In [16]:
selection_summary = selector.get_selection_summary()
print('Feature được giữ sau Mutual Information:')
display(selection_summary.loc[selection_summary['selected']].reset_index(drop=True))

Feature được giữ sau Mutual Information:


,feature,mutual_information,discrete,selected
0,robust_numeric__pdays,0.058789,False,True
1,onehot_nominal__poutcome_success,0.046488,True,True
2,robust_engineered__successful_prior_contact_re...,0.041693,False,True
3,robust_engineered__prior_contact_recency_log,0.041246,False,True
4,robust_engineered__previous_contact_intensity,0.039020,False,True
5,robust_numeric__balance,0.037471,False,True
6,robust_engineered__balance_per_age,0.036438,False,True
7,engineered_binary__contact_known,0.034639,True,True
8,onehot_nominal__contact_missing,0.034639,True,True
9,robust_engineered__campaign_vs_previous_ratio,0.034548,False,True


### Kết luận

- Dữ liệu được chia stratified 80/20 trước mọi thao tác học thống kê: 8.929 dòng train và 2.233 dòng test.
- Pipeline tích hợp theo thứ tự: schema/leakage guard → cleaning → 24 engineered features → encoding/scaling → một MI selector.
- `duration` bị loại trước cleaning/feature engineering và output bất biến với giá trị của cột này.
- Số chiều: 16 predictor thô → 15 predictor pre-call → 39 feature sau engineering → 74 cột sau encoding → 32 cột sau MI.
- Train/test có cùng schema, không còn NaN hoặc infinity; pipeline và bốn Parquet reload thành công.
- Notebook dừng tại preprocessing, không huấn luyện hoặc đánh giá mô hình.